# STR Factor Backtest Notebook

本 Notebook 展示如何使用 Akshare 拉取 A 股数据，计算 20 日换手率稳定性因子（STR），并基于 T+1 原则进行分组回测。所有注释均使用中文，图表的标题和图例使用英文，以避免本地缺少中文字体导致的渲染问题。

In [ ]:
# 安装与导入依赖库
# 如果本地已安装可跳过安装行，为了保证 Notebook 可运行，这里保留安装指令。
# 使用中文注释解释每一步的作用。
# 安装 Akshare（如已安装可注释掉下一行）
# !pip install -U akshare

import os
import math
import datetime as dt
import pandas as pd
import numpy as np

import time
import requestsimport akshare as ak
import seaborn as sns
import matplotlib.pyplot as plt

# 设置绘图风格，确保图表美观。
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# 定义全局输出目录，存放图片等文件。
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "outputs")
IMAGE_DIR = os.path.join(OUTPUT_DIR, "images")

# 创建输出目录，exist_ok=True 避免重复创建时报错。
os.makedirs(IMAGE_DIR, exist_ok=True)
print(f"输出目录: {IMAGE_DIR}")

In [ ]:
# 数据抓取相关函数
# 这些函数负责从 Akshare 拉取股票列表、日行情与指数数据。

# 设定回测日期范围，便于在多个函数中复用。
START_DATE = "20180101"  # Akshare 接口一般要求 yyyyMMdd 格式
END_DATE = "20241231"
INDEX_SYMBOL = "000905"  # 中证 500

def fetch_stock_basic() -> pd.DataFrame:
    """
    获取 A 股股票列表，并过滤掉退市股票。
    返回包含股票代码与名称的 DataFrame。
    """
    # 使用 akshare 提供的股票列表接口，拉取基础信息。
    stock_basic = ak.stock_info_a_code_name()
    # 删除可能的空值，确保代码字段完整。
    stock_basic = stock_basic.dropna(subset=['code'])
    stock_basic = stock_basic.rename(columns={'code': 'symbol', 'name': 'display_name'})
    print(f"共获取股票数量: {len(stock_basic)}")
    return stock_basic


def fetch_daily_kline(symbol: str, max_retry: int = 3, sleep_secs: float = 1.0) -> pd.DataFrame:
    """
    拉取单只股票在指定区间的日行情，包含收盘价、换手率、市值等字段。
    增加简单的重试与间隔逻辑，防止网络抖动或目标站点限流导致的连接中断。
    """
    for attempt in range(1, max_retry + 1):
        try:
            df = ak.stock_zh_a_hist(
                symbol=symbol,
                period="daily",
                start_date=START_DATE,
                end_date=END_DATE,
                adjust="qfq",
                timeout=15
            )
            break
        except requests.exceptions.RequestException as e:
            print(f"[警告] 第 {attempt} 次拉取 {symbol} 失败: {e}")
            if attempt < max_retry:
                time.sleep(sleep_secs)
            else:
                print(f"[错误] 多次尝试仍失败，跳过 {symbol}")
                return pd.DataFrame()
    if df.empty:
        return df
    df = df.rename(columns={
        '日期': 'date',
        '收盘': 'close',
        '开盘': 'open',
        '最高': 'high',
        '最低': 'low',
        '成交量': 'volume',
        '成交额': 'amount',
        '换手率': 'turnover_rate',
        '流通市值': 'float_mv',
        '总市值': 'total_mv'
    })
    df['date'] = pd.to_datetime(df['date'])
    df['symbol'] = symbol
    return df
def fetch_index_quote(symbol: str = INDEX_SYMBOL) -> pd.DataFrame:
    """
    获取指数日行情，用于基准对比（例如中证 500）。
    返回的 DataFrame 包含 date 和 close 两列。
    """
    index_df = ak.index_zh_a_hist(symbol=symbol, period="daily", start_date=START_DATE, end_date=END_DATE, adjust="")
    index_df = index_df.rename(columns={'日期': 'date', '收盘': 'close'})
    index_df['date'] = pd.to_datetime(index_df['date'])
    index_df = index_df[['date', 'close']].sort_values('date')
    index_df = index_df.set_index('date').sort_index()
    print(f"指数 {symbol} 数据行数: {len(index_df)}")
    return index_df

In [ ]:
# 数据预处理函数

def prepare_panel_data(stock_basic: pd.DataFrame, max_stock: int = 200) -> pd.DataFrame:
    """
    将多只股票的日行情数据合并为一张长表。
    参数 max_stock 用于限制样本量，避免一次性拉取全部股票导致时间过长。
    返回包含 date、symbol、close、turnover_rate、float_mv 等列的数据表。
    """
    records = []
    # 只取前 max_stock 只股票作为示例，实际回测可移除此限制。
    for i, row in stock_basic.head(max_stock).iterrows():
        code = row['symbol']
        print(f"拉取 {code} 数据...")
        daily_df = fetch_daily_kline(code)

        # 每只股票之间暂停一小段时间，降低请求频率，避免触发限流。
        time.sleep(0.5)
        if daily_df.empty:
            print(f"{code} 无数据，跳过")
            continue
        records.append(daily_df)
    if not records:
        return pd.DataFrame()
    merged = pd.concat(records, ignore_index=True)
    # 剔除缺失值，保持字段完整。
    merged = merged.dropna(subset=['close', 'turnover_rate'])
    # 仅保留需要的字段。
    merged = merged[['date', 'symbol', 'close', 'turnover_rate', 'float_mv']]
    print(f"合并后总行数: {len(merged)}")
    return merged


def filter_investable_universe(df: pd.DataFrame, min_history: int = 60) -> pd.DataFrame:
    """
    清洗数据，剔除上市未满 min_history 日的股票，并确保时间顺序正确。
    """
    df = df.sort_values(['symbol', 'date'])
    # 计算每只股票的上市天数（基于数据开始日期）。
    df['days_from_start'] = df.groupby('symbol').cumcount() + 1
    df = df[df['days_from_start'] >= min_history].copy()
    # 过滤掉换手率或市值为 0 的异常值。
    df = df[(df['turnover_rate'] > 0) & (df['float_mv'] > 0)]
    return df

In [ ]:
# 因子计算函数

from typing import Tuple

def compute_str_factor(df: pd.DataFrame, window: int = 20) -> pd.DataFrame:
    """
    计算 STR 因子：对每只股票的换手率做 rolling std，然后在截面上对对数流通市值做回归中性化。
    返回包含 date、symbol、str_raw、str_factor 的表格。
    """
    # 先按股票与日期排序，确保 rolling 计算正确。
    df = df.sort_values(['symbol', 'date']).copy()
    # 计算 rolling 标准差，刻画换手率的稳定性。
    df['str_raw'] = df.groupby('symbol')['turnover_rate'].transform(lambda x: x.rolling(window).std())

    # 删除不足窗口的数据，避免前期大量 NaN。
    df = df.dropna(subset=['str_raw'])

    # 在每个交易日横截面上，对 str_raw 与对数市值进行回归，提取残差作为中性化因子值。
    factor_list = []
    for date, daily in df.groupby('date'):
        # 计算对数市值，避免数据尺度过大。
        daily = daily.copy()
        daily['ln_mktcap'] = np.log(daily['float_mv'])
        # 使用最小二乘回归：str_raw = alpha + beta * ln_mktcap + eps
        x = daily['ln_mktcap']
        y = daily['str_raw']
        # 添加常数项 1 方便计算系数。
        X = np.vstack([np.ones_like(x), x]).T
        beta_hat = np.linalg.lstsq(X, y, rcond=None)[0]
        alpha, beta = beta_hat
        # 计算残差作为最终因子值。
        daily['str_factor'] = y - (alpha + beta * x)
        factor_list.append(daily[['date', 'symbol', 'str_raw', 'str_factor']])

    factor_df = pd.concat(factor_list, ignore_index=True)
    return factor_df


def merge_with_returns(factor_df: pd.DataFrame, price_df: pd.DataFrame) -> pd.DataFrame:
    ""
    将因子表与下期收益率对齐，遵循 T+1 原则。
    price_df 需要包含 date、symbol、close。
    ""
    # 按股票与日期排序，计算下一交易日的收益率。
    price_df = price_df.sort_values(['symbol', 'date']).copy()
    price_df['next_close'] = price_df.groupby('symbol')['close'].shift(-1)
    price_df['next_ret'] = price_df['next_close'] / price_df['close'] - 1

    merged = pd.merge(factor_df, price_df[['date', 'symbol', 'next_ret']], on=['date', 'symbol'], how='left')
    # 删除未来收益缺失的数据点（通常为样本末尾）。
    merged = merged.dropna(subset=['next_ret'])
    return merged

In [ ]:
# 分组与回测相关函数

def build_factor_groups(factor_df: pd.DataFrame, n_group: int = 10) -> pd.DataFrame:
    ""
    在每个交易日内按因子值分为 n_group 个等人数分组，返回包含 group_id 的表。
    ""
    grouped_list = []
    for date, daily in factor_df.groupby('date'):
        daily = daily.copy()
        # 使用 qcut 做等分分组，duplicates='drop' 避免因重复值导致分组失败。
        daily['group'] = pd.qcut(daily['str_factor'], q=n_group, labels=False, duplicates='drop')
        grouped_list.append(daily)
    grouped_df = pd.concat(grouped_list, ignore_index=True)
    return grouped_df


def compute_group_returns(group_df: pd.DataFrame, n_group: int = 10) -> Tuple[pd.DataFrame, pd.DataFrame]:
    ""
    计算每个分组的日度等权收益，并输出累计净值（乘法方式）。
    返回：日度收益表 daily_ret_pivot 与净值表 nav_pivot。
    ""
    # 计算当日分组在下一交易日的等权收益率。
    daily_group_ret = group_df.groupby(['date', 'group'])['next_ret'].mean().reset_index()
    daily_ret_pivot = daily_group_ret.pivot(index='date', columns='group', values='next_ret').sort_index()
    # 乘法累计，初始净值为 1。
    nav_pivot = (1 + daily_ret_pivot).cumprod()
    nav_pivot = nav_pivot.rename(columns=lambda x: f"Group_{int(x)}")
    return daily_ret_pivot, nav_pivot


def compute_ic_icir(group_df: pd.DataFrame) -> pd.DataFrame:
    ""
    计算日度 IC 序列及其均值、标准差、ICIR。
    返回包含 ic_mean、ic_std、ic_ir 的单行 DataFrame。
    ""
    ic_series = group_df.groupby('date').apply(lambda x: x['str_factor'].corr(x['next_ret']))
    ic_mean = ic_series.mean()
    ic_std = ic_series.std(ddof=1)
    ic_ir = ic_mean / ic_std * math.sqrt(252) if ic_std and not math.isnan(ic_std) else np.nan
    summary = pd.DataFrame({
        'ic_mean': [ic_mean],
        'ic_std': [ic_std],
        'ic_ir': [ic_ir]
    })
    return summary, ic_series

In [ ]:
# 作图与结果输出函数

def plot_nav(nav_df: pd.DataFrame, title: str, filename: str):
    ""
    绘制累计净值曲线，标题与图例使用英文，图像保存为 PNG。
    ""
    plt.figure(figsize=(12, 6))
    for col in nav_df.columns:
        plt.plot(nav_df.index, nav_df[col], label=col)
    plt.title(title)
    plt.xlabel('Date')
    plt.ylabel('Net Asset Value')
    plt.legend()
    plt.tight_layout()
    save_path = os.path.join(IMAGE_DIR, filename)
    plt.savefig(save_path, dpi=200, format='png')
    plt.show()
    print(f"图已保存: {save_path}")


def plot_ic(ic_series: pd.Series, title: str, filename: str):
    ""
    绘制 IC 时间序列曲线，展示因子预测能力的稳定性。
    ""
    plt.figure(figsize=(12, 4))
    plt.plot(ic_series.index, ic_series.values, color='steelblue')
    plt.axhline(0, color='red', linestyle='--', linewidth=1)
    plt.title(title)
    plt.xlabel('Date')
    plt.ylabel('IC')
    plt.tight_layout()
    save_path = os.path.join(IMAGE_DIR, filename)
    plt.savefig(save_path, dpi=200, format='png')
    plt.show()
    print(f"IC 图已保存: {save_path}")


def plot_vs_index(nav_df: pd.DataFrame, index_nav: pd.Series, title: str, filename: str):
    ""
    将第一组与最后一组的净值与指数进行对比，便于观察超额收益。
    ""
    plt.figure(figsize=(12, 6))
    # 仅选取代表性的分组（最低与最高因子值）。
    if 'Group_0' in nav_df.columns:
        plt.plot(nav_df.index, nav_df['Group_0'], label='Group_0 (Low STR)')
    if nav_df.columns.size > 1:
        last_col = nav_df.columns[-1]
        plt.plot(nav_df.index, nav_df[last_col], label=f'{last_col} (High STR)')
    plt.plot(index_nav.index, index_nav.values, label='CSI 500 Index', linestyle='--', color='black')
    plt.title(title)
    plt.xlabel('Date')
    plt.ylabel('Net Asset Value')
    plt.legend()
    plt.tight_layout()
    save_path = os.path.join(IMAGE_DIR, filename)
    plt.savefig(save_path, dpi=200, format='png')
    plt.show()
    print(f"指数对比图已保存: {save_path}")

In [ ]:
# 主流程示例
# 这个单元整合前面的函数，串联完成数据抓取、因子计算、分组回测与作图。

# 1) 获取股票基础列表并抓取行情数据（如需全市场，可去掉 max_stock 限制）。
stock_basic = fetch_stock_basic()
price_panel = prepare_panel_data(stock_basic, max_stock=50)  # 为控制示例耗时，仅取 50 只股票

if price_panel.empty:
    print("未获取到有效行情数据，后续步骤无法执行。")
else:
    # 2) 基础清洗，剔除上市不满 60 日的样本，避免初期噪声。
    clean_panel = filter_investable_universe(price_panel, min_history=60)
    print(f"清洗后数据行数: {len(clean_panel)}")

    # 3) 计算 STR 因子并与未来收益对齐。
    factor_df = compute_str_factor(clean_panel, window=20)
    factor_with_ret = merge_with_returns(factor_df, clean_panel[['date', 'symbol', 'close']])
    print(f"带有未来收益的样本量: {len(factor_with_ret)}")

    # 4) 分组并计算分组收益与净值。
    grouped_factor = build_factor_groups(factor_with_ret, n_group=10)
    daily_ret, nav = compute_group_returns(grouped_factor, n_group=10)

    # 5) 计算 IC 与 ICIR。
    ic_summary, ic_series = compute_ic_icir(grouped_factor)
    print("IC 汇总:")
    print(ic_summary)

    # 6) 获取指数基准并转换为净值曲线（乘法累计）。
    index_df = fetch_index_quote(INDEX_SYMBOL)
    index_df['ret'] = index_df['close'].pct_change()
    index_nav = (1 + index_df['ret'].fillna(0)).cumprod()

    # 7) 绘制净值、IC、指数对比图，保存为 PNG。
    plot_nav(nav, title="STR Group NAV", filename="str_group_nav.png")
    plot_ic(ic_series, title="IC Series", filename="str_ic_series.png")
    plot_vs_index(nav, index_nav, title="Group vs CSI500", filename="group_vs_csi500.png")

## 使用说明

1. 运行 Notebook 前请确保已安装 `akshare` 并能正常联网以抓取数据。
2. 如需全市场回测，将 `prepare_panel_data` 中的 `max_stock` 参数调大或移除，但请注意运行耗时与内存占用。
3. 图表文件会保存到 `../outputs/images/` 目录，文件格式为 PNG。
4. 若本地缺少中文字体，可保持当前设置：注释中文、图表标题与图例英文，可避免字体报错。